# Introduction

# Stage 03 — Main results, and the artifact test

**Pipeline position:** third. Reads stage 02's row-level frame; writes the full summary grid.

## What this stage answers

**Does the agreement signal work — and if the number looks large, is it measuring what it appears
to?**

Two cells, in that order, and the order is the argument:

1. **The primary table.** Pooled 2024–2025, both rank universes, both populations, all four
   thresholds. Every summary cell in the study is also built here — eight panels x four thresholds x
   two universes x two populations x five split types — and exported.
2. **The artifact test.** The full-population result is large. Before reporting it, this stage tests
   the alternative explanation directly: that deep ADP is not a real price, and two competent
   projections disagreeing with a near-random ordering will both be right for reasons that have
   nothing to do with beating a market.

Running the artifact test *after* seeing the headline, rather than instead of it, is deliberate. The
headline is reported in full; then it is interrogated.

## What to watch

In the primary table, read `median_adp_overall` **beside** `hit_rate`, not after it. If a cell's
median draft price sits deep in the hundreds, its players are not underpriced — they have no
meaningful price, and "beating ADP" for them is a different claim.

## Inputs and outputs

| Direction | Path |
|---|---|
| in | `artifacts/player_season_results.csv`, `00_shared_pipeline.ipynb` |
| out | `artifacts/threshold_summary.csv` (Sleeper-ADP rows; stage 06 appends the Underdog rows) |

### Explain — load the shared library

Every stage notebook begins here. It loads `00_shared_pipeline.ipynb` using the repo's convention
(`memory/prefer-ipynb-not-py.md`, mirroring the loader in `betting/predict_totals.ipynb` cell 4):
**json + exec over the library's code cells**, never `%run` (brittle across nbclient / papermill /
VSCode) and never a `.py` module (the repo is notebook-centric by rule).

`RUN_TESTS = False` and `SHARED_VERBOSE = False` are set **before** the exec, so the library's inline
tests are skipped and its configuration banner stays silent — those belong to a standalone run of the
library, not to every consumer.

The cell prints a compact load record: the SHA-256 of the library notebook itself, the count of names
imported, and the pinned parameters. Recording the library's hash means each stage's output states
exactly which version of the shared code produced it — if the library changes, the stages' recorded
hashes diverge and the mismatch is visible rather than silent.

In [1]:
import json as _json
from pathlib import Path as _Path


def _exec_notebook(path, glob):
    """Execute every code cell of a notebook into `glob` (repo convention: json + exec)."""
    with open(path, encoding="utf-8") as _fh:
        _nb = _json.load(_fh)
    for _cell in _nb["cells"]:
        if _cell["cell_type"] == "code":
            exec("".join(_cell["source"]), glob)


RUN_TESTS = False          # skip the library's inline self-tests in a consumer
SHARED_VERBOSE = False     # suppress the library's configuration banner
_SHARED = "00_shared_pipeline.ipynb"
_before = set(globals())
_exec_notebook(_SHARED, globals())
_loaded = sorted(n for n in set(globals()) - _before
                 if not n.startswith("_") and n not in {"RUN_TESTS", "SHARED_VERBOSE"})

print(f"loaded {_SHARED}")
print(f"  library sha256 : {sha256_file(_SHARED)}")
print(f"  names imported : {len(_loaded)}")
print(f"  functions      : {[n for n in _loaded if callable(globals()[n])]}")
print(f"  repo           : {REPO.name}   project: {PROJECT.name}")
print(f"  seasons {TEST_SEASONS} | thresholds {THRESHOLDS} | populations {list(POPULATIONS)}")
print(f"  seed {SEED} | perms {N_PERM:,} | boots {N_BOOT:,}")

loaded 00_shared_pipeline.ipynb
  library sha256 : d3e28a60fab75caf19c5387de293057de842c21c3c088edbc204acd89972b469
  names imported : 48
  functions      : ['Path', 'add_signals', 'boot_index_matrix', 'bootstrap_lift', 'build_ranks', 'canonical_strata', 'correct_vec', 'datetime', 'logistic_design', 'logistic_newton', 'norm', 'perm_sign_matrix', 'permutation_test', 'population_slice', 'sha256_file', 'spearmanr', 'summarise_cell', 'thr_col', 'timezone', 'wilson']
  repo           : JoSchoAnalytics   project: adp_consensus_agreement_2026-08-02
  seasons [2021, 2022, 2023, 2024, 2025] | thresholds [0.0, 5.0, 7.5, 10.0] | populations ['all_adp', 'drafted_top180']
  seed 20260802 | perms 10,000 | boots 10,000


### Interpretation — library loaded, this stage is anchored to it

The load record confirms the shared library executed cleanly and lists the names now in scope,
including the analysis functions this stage calls. The pinned parameters match the study's
declaration — seasons 2021–2025, thresholds `[0, 5, 7.5, 10]`, both populations, seed 20260802 — so
this notebook cannot silently disagree with its siblings about what a rank is or how a hit rate is
scored.

The **library SHA-256 is printed and recorded**. Every stage prints the same digest, which is what
makes "all seven stages ran against the same library" a checkable claim rather than an assumption;
stage 07 re-hashes the library and compares.

`RUN_TESTS=False` means the library's self-tests did not run here — they belong to a standalone run
of `00_shared_pipeline.ipynb`, which is the gate for this pipeline being trustworthy at all.

**This stage reads:** `artifacts/player_season_results.csv` and the shared library
**and writes:** `artifacts/threshold_summary.csv` (the 1,298 Sleeper-ADP rows)

### Explain — build the full summary grid and print the primary table

This cell reads stage 02's row-level frame, builds **every** summary cell in the study, and prints the
primary panel.

**The grid.** Two populations x two universes x eight panels x four thresholds x five split types
(overall, direction, position, season, veteran/rookie). All of it computed and all of it exported —
which matters for a post-hoc study, because it removes any room to report only a favourable slice.

**The primary table** is pooled 2024–2025, both universes, both populations. Columns: `n` agreement
calls; `hits`/`misses`/`ties` partitioning it; `hit_rate` counting ties as misses; the Wilson 95%
interval; `hit_rate_ex_ties`; mean and median `actual_gap` in rank spots; the Spearman correlation
between agreement strength and realised move; and **`median_adp_overall`**.

**How to read it, and the trap.** The instinct is to read down `hit_rate` and conclude the signal
strengthens with the threshold. Before doing that, read `median_adp_overall` **beside** it. If a
cell's median draft price sits deep in the hundreds, those players are not underpriced — they have no
meaningful price, and "beating ADP" for them is a different claim. The next cell tests that directly.

**Universe A vs B** differ only in whether Sleeper-less rows sit in the rank denominators. If
conclusions move between them, the result is population-sensitive; if not, that axis is settled and
the remaining variation is attributable elsewhere.

In [2]:
PLAYER_RESULTS = pd.read_csv(ARTIFACTS / "player_season_results.csv")
SIGNALS = PLAYER_RESULTS.rename(columns={"position": "pos", "model_pred": "pred",
                                         "actual_half_ppr": "y", "model_family": "model"})
print(f"loaded artifacts/player_season_results.csv: {len(SIGNALS):,} rows")
assert len(SIGNALS) == 5315, f"unexpected row count {len(SIGNALS)}"

_summary_rows = []
for pop_name in POPULATIONS:
    for uni in ("A", "B"):
        d_all = SIGNALS[(SIGNALS.population == pop_name) & (SIGNALS.universe == uni) & SIGNALS.complete]
        for panel, seasons in PANELS.items():
            panel_df = d_all[d_all.season.isin(seasons)]
            for t in THRESHOLDS:
                cell = panel_df[panel_df[thr_col(t)]]
                base = {"market": "sleeper_adp", "population": pop_name, "universe": uni,
                        "panel": panel, "threshold": t, "panel_complete_n": len(panel_df)}
                _summary_rows.append(summarise_cell(cell, {**base, "split": "all", "split_value": "all"}))
                for split_name, col in (("direction", "direction"), ("position", "pos"),
                                        ("season", "season"), ("group", "group")):
                    for val, grp in cell.groupby(col):
                        _summary_rows.append(summarise_cell(
                            grp, {**base, "split": split_name, "split_value": str(val)}))
SUMMARY = pd.DataFrame(_summary_rows)

_SHOW = ["n", "hits", "misses", "ties", "hit_rate", "wilson_lo", "wilson_hi", "hit_rate_ex_ties",
         "mean_actual_gap", "median_actual_gap", "spearman_consensus_vs_actual_gap", "median_adp_overall"]
for pop_name, tag in (("all_adp", "FULL ADP-BEARING POPULATION"),
                      ("drafted_top180", f"DRAFTED BOARD (adp_overall_rank <= {DRAFTABLE_POOL_SIZE})")):
    t = SUMMARY[(SUMMARY.panel == "pooled_2024_2025") & (SUMMARY.population == pop_name)
                & (SUMMARY.split == "all")].sort_values(["universe", "threshold"])
    print("=" * 120)
    print(f"POOLED 2024-2025 — {tag}   (complete rows in panel: {int(t.panel_complete_n.iloc[0]):,})")
    print("=" * 120)
    print(t.set_index([t.universe, t.threshold])[_SHOW].round(4).to_string())
    print()

SUMMARY.to_csv(ARTIFACTS / "threshold_summary.csv", index=False)
print(f"SUMMARY built and written: {len(SUMMARY):,} rows -> artifacts/threshold_summary.csv")
print(f"  splits: {sorted(SUMMARY.split.unique())} | panels: {SUMMARY.panel.nunique()} "
      f"| thresholds: {sorted(SUMMARY.threshold.unique())}")
print("  (stage 06 appends the dated-Underdog rows to this same file)")

loaded artifacts/player_season_results.csv: 5,315 rows


POOLED 2024-2025 — FULL ADP-BEARING POPULATION   (complete rows in panel: 866)
                      n  hits  misses  ties  hit_rate  wilson_lo  wilson_hi  hit_rate_ex_ties  mean_actual_gap  median_actual_gap  spearman_consensus_vs_actual_gap  median_adp_overall
universe threshold                                                                                                                                                                     
A        0.0        512   409      97     6    0.7988     0.7619     0.8313            0.8083           9.0938                5.0                            0.7636              459.55
         5.0        338   300      37     1    0.8876     0.8494     0.9170            0.8902          16.1538               14.0                            0.7867              634.25
         7.5        307   277      29     1    0.9023     0.8639     0.9307            0.9052          17.4235               16.0                            0.7878              640.70
 

### Interpretation — a very large headline, and a column that undermines it

**Full ADP-bearing population, universe A, pooled 2024–2025:** the agreement cell hits **79.9%**
(409/512) at t>0, rising monotonically to **88.8%** (338), **90.2%** (307) and **92.3%** (259). Wilson
intervals are tight and nowhere near 0.50. Spearman between agreement strength and realised move runs
+0.76 to +0.79, so bigger agreements do correspond to bigger moves. Read alone, this looks like a
decisive market-beating signal.

**Now read `median_adp_overall` in the same rows: 459.6, 634.3, 640.7, 652.0.** The median agreement
call at `t>5` is a player drafted, on average, at **pick 634**. In a 12-team, 15-round league only
about 180 players are drafted at all. These are not underpriced players; they are players with no
meaningful price. And the metric rises with the threshold *while the median ADP rises alongside it* —
exactly the pattern an artifact produces. The next cell tests it rather than leaving it an impression.

**Drafted board, same panel and universe:** **66.1%** (107/162) at t>0, then **85.0%** (n=40),
**93.1%** (n=29), **100%** (15/15), with median ADP staying at 97–131 — inside the draftable range, so
these are calls on players people actually draft. But the sample collapses fourfold, and the Wilson
interval at t>10 is **[0.796, 1.000]**: fifteen consecutive correct calls is genuinely unlikely by
chance, and also only fifteen calls.

**Universe A vs B changes nothing.** On the full population B runs 1–3 points lower at every threshold
(77.6 / 85.4 / 87.0 / 88.7); on the drafted board B is within 2 points everywhere and identical at
t>10. No conclusion flips. **The rank-universe axis is settled and can be set aside** — the remaining
variation comes from the drafted/undrafted axis and from sample size.

One artifact of universe A worth noting: `mean_actual_gap` is much larger there than in B on the full
population (+9.1 vs +1.4 at t>0), because A's denominators include the 204 Sleeper-less rows so the
same player can move more rank spots. Compare gap magnitudes only within a universe.

### Explain — test the alternative explanation directly

The previous cell produced a very high hit rate on the full population. A result that large against a
market price deserves interrogation before celebration, so this cell tests the competing explanation.

**The hypothesis.** Beyond roughly the 180th pick, "average draft position" is an artifact of a
handful of drafts in a very long tail and carries almost no information about expected production. Two
competent projections that both disagree with a near-random ordering will both be right, and will
agree with each other while doing it — producing a high agreement hit rate that has nothing to do with
beating a market.

**Four pieces of evidence.**

1. **ADP depth of the agreement cell versus the panel it is drawn from**, at each threshold. If the
   cell's ADP distribution shifts progressively deeper as the threshold rises, then "the signal
   strengthens with the threshold" is really "the cell becomes more undrafted".
2. **The share of each cell sitting outside the draftable top 180.**
3. **Hit rate split inside versus outside** that boundary — the direct comparison, with realised
   season totals alongside.
4. **What the largest calls actually scored.** If the biggest wins are players who scored a handful of
   points all year, the cell is ordering noise.

**What would refute the hypothesis:** roughly equal hit rates inside and outside, and agreement calls
whose median ADP sits inside the drafted range.

In [3]:
print("ADP DEPTH OF THE AGREEMENT CELL — full population, universe A, pooled 2024-2025")
print("=" * 112)
_panel = SIGNALS[(SIGNALS.population == "all_adp") & (SIGNALS.universe == "A")
                 & SIGNALS.complete & SIGNALS.season.isin([2024, 2025])]
_rows = [{"scope": "whole panel", "n": len(_panel), "adp_p25": _panel.adp_half_ppr.quantile(.25),
          "adp_median": _panel.adp_half_ppr.median(), "adp_p75": _panel.adp_half_ppr.quantile(.75),
          "pct_outside_top180": (_panel.adp_overall_rank > DRAFTABLE_POOL_SIZE).mean(), "hit_rate": np.nan}]
for t in THRESHOLDS:
    c = _panel[_panel[thr_col(t)]]
    _rows.append({"scope": f"agreement t>{t:g}", "n": len(c), "adp_p25": c.adp_half_ppr.quantile(.25),
                  "adp_median": c.adp_half_ppr.median(), "adp_p75": c.adp_half_ppr.quantile(.75),
                  "pct_outside_top180": (c.adp_overall_rank > DRAFTABLE_POOL_SIZE).mean(),
                  "hit_rate": (c.outcome == "hit").mean()})
print(pd.DataFrame(_rows).round(3).to_string(index=False))

print("\nHIT RATE INSIDE vs OUTSIDE the draftable top 180 (same panel, same agreement cells)")
print("-" * 112)
_split = []
for t in THRESHOLDS:
    c = _panel[_panel[thr_col(t)]].copy()
    c["zone"] = np.where(c.adp_overall_rank <= DRAFTABLE_POOL_SIZE, "inside_top180", "outside_top180")
    for zone, g in c.groupby("zone"):
        lo, hi = wilson(int((g.outcome == "hit").sum()), len(g))
        _split.append({"threshold": t, "zone": zone, "n": len(g), "hits": int((g.outcome == "hit").sum()),
                       "hit_rate": (g.outcome == "hit").mean(), "wilson_lo": lo, "wilson_hi": hi,
                       "median_adp": g.adp_half_ppr.median(), "median_actual_pts": g.y.median()})
print(pd.DataFrame(_split).round(3).to_string(index=False))

print("\nTHE LARGEST FULL-POPULATION AGREEMENT CALLS (t>5) AND WHAT THEY ACTUALLY SCORED")
print("-" * 112)
_big = _panel[_panel[thr_col(5.0)]].nlargest(10, "consensus_score")
print(_big[["season", "pos", "player", "adp_half_ppr", "adp_rank", "model_rank", "sleeper_rank",
            "actual_rank", "consensus_score", "actual_gap", "y", "outcome"]]
      .rename(columns={"y": "actual_pts", "adp_half_ppr": "adp_overall"}).round(1).to_string(index=False))
print(f"\n  median season total of those 10 calls : {_big.y.median():.1f} half-PPR points")
print(f"  median for a top-180 drafted player   : "
      f"{_panel[_panel.adp_overall_rank <= DRAFTABLE_POOL_SIZE].y.median():.1f}")

ADP DEPTH OF THE AGREEMENT CELL — full population, universe A, pooled 2024-2025
          scope   n  adp_p25  adp_median  adp_p75  pct_outside_top180  hit_rate
    whole panel 866  111.125      258.30  643.875               0.597       NaN
  agreement t>0 512  177.600      459.55  671.625               0.697     0.799
  agreement t>5 338  286.550      634.25  682.200               0.861     0.888
agreement t>7.5 307  296.550      640.70  685.250               0.879     0.902
 agreement t>10 259  412.900      652.00  687.200               0.919     0.923

HIT RATE INSIDE vs OUTSIDE the draftable top 180 (same panel, same agreement cells)
----------------------------------------------------------------------------------------------------------------
 threshold           zone   n  hits  hit_rate  wilson_lo  wilson_hi  median_adp  median_actual_pts
       0.0  inside_top180 155   103     0.665      0.587      0.734       95.40             120.28
       0.0 outside_top180 357   306     0.85

### Interpretation — confirmed: the full-population headline is an undrafted-tail artifact

The first table settles it. The whole 2024–2025 panel has a median ADP of 258 and is 59.7% outside the
draftable 180. The **agreement cells are progressively more undrafted than the panel they are drawn
from**: 69.7% outside at t>0, then **86.1%, 87.9% and 91.9%**. Median ADP climbs from 459.6 to
**652.0**. The apparent "signal strengthens with threshold" is, to a first approximation, "the cell
becomes almost entirely undrafted".

The inside/outside split shows where the effect actually lives:

| t | inside top-180 | outside top-180 |
|---|---|---|
| 0 | **66.5%** (n=155) | **85.7%** (n=357) |
| >5 | 85.1% (n=47) | 89.3% (n=291) |
| >7.5 | 89.2% (n=37) | 90.4% (n=270) |
| >10 | 95.2% (n=21) | 92.0% (n=238) |

**This is more nuanced than a blanket dismissal, and both halves must be stated.** At t>0 the gap is
enormous — 19 points — and the pooled 79.9% is essentially a weighted average dragged upward by 357
undrafted calls. At higher thresholds the two zones **converge**, and at t>10 the inside rate is
actually *higher*. So the artifact is not that undrafted calls are easy at every threshold; it is
that **the pooled full-population number is dominated by a subpopulation making up 86–92% of the cell
and answering a different question**. Quoting 92.3% as a market-beating result is still wrong, because
the number describes a cell that is nine-tenths players nobody drafts.

The realised-points column removes any doubt about what "correct" means out there: median season total
**25.0 half-PPR points outside** the top 180 against **120.3 inside**. And the ten largest calls —
Tutu Atwell (28.2 points), Nick Westbrook-Ikhine (14.4), Tai Felton (4.0), Allen Lazard (18.0) — have
a median season total of **57.2** against **136.4** for a typical drafted player. Both projections
correctly ordered the noise floor. That is a true statement about deep ADP being uninformative, not a
claim about beating a market.

**Consequence for the rest of the pipeline:** the drafted board is the decision-relevant population.
The full population stays in every table for completeness — and because the contrast is itself the
finding — but no headline is taken from it.

# Conclusion and Next Steps

## What this stage established

**The headline, reported in full.** Full ADP-bearing population, universe A, pooled 2024–2025:
**79.9%** (409/512) at t>0, rising to **88.8%**, **90.2%** and **92.3%** at t>5, t>7.5 and t>10.
Wilson intervals tight and nowhere near 0.50.

**And the artifact test, which reframes it.** The agreement cells are progressively more undrafted
than the panel they are drawn from — **69.7% → 86.1% → 87.9% → 91.9%** outside the draftable top 180
— with median ADP climbing from 459.6 to **652.0**. Median realised season total outside the top 180
is **25.0 half-PPR points** against **120.3** inside. The ten largest full-population calls have a
median season total of **57.2** against **136.4** for a typical drafted player.

**The nuance, which matters and must not be flattened.** The inside/outside split is *not* uniform
across thresholds. At t>0 the gap is enormous — **66.5% (n=155) inside vs 85.7% (n=357) outside** —
but the two zones **converge by t>7.5** (89.2% vs 90.4%) and **invert at t>10** (95.2% vs 92.0%). So
the artifact is not "undrafted calls are easy at every threshold". It is that **the pooled
full-population number is dominated by a subpopulation making up 86–92% of the cell and answering a
different question.**

**The drafted board, which is the decision-relevant result.** **66.1%** (n=162), **85.0%** (n=40),
**93.1%** (n=29), **100%** (15/15), with median ADP staying at 97–131.

**Universe A vs B changes nothing.** B runs 1–3 points lower on the full population and within 2
points on the drafted board, identical at t>10. **The rank-universe axis is settled.** Every
remaining conclusion moves along the drafted/undrafted axis and with sample size.

## What is now true

`artifacts/threshold_summary.csv` holds 1,298 Sleeper-ADP summary cells. Stage 06 appends the
Underdog rows; stage 07 reconstructs every one of them from the row-level file.

## Next step

Run **`04_stability.ipynb`** — a headline on one two-season window is not a result until it is shown
to hold across seasons, positions and player groups.